In [1]:
import re
import pandas as pd
import ast
from urllib.parse import quote_plus
from pathlib import Path 

from selenium import webdriver
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC 
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
import time
import json

We're converting the Hotel RevPar table into a CSV that will be used by the Google Scraper.


In [3]:
# adjust the CSV path as needed 
SRC_FILE = Path("hotels.csv")
DEST_FILE = SRC_FILE.with_suffix(".cleaned.csv")
SNOWFLAKE_READY_FILE = SRC_FILE.with_suffix(".snowflake_ready.csv")


In [3]:
def clean_htl_key(key: str) -> str:
    """Remove trailing dots from the HTL_Key value."""
    if pd.isna(key):
        return ""
    return str(key).rstrip(".")

def to_search_query(text: str) -> str:
    """Convert General Info into a safe URL segment (query)."""
    if pd.isna(text):
        return ""
    # keep only [A-Za-z0-9 ], remove extra spaces, trim,  lowercase, encode
    cleaned = re.sub(r"[^A-Za-z0-9\s]", " ", text)
    cleaned = re.sub(r"\s+", " ", cleaned).strip().lower()
    return quote_plus(cleaned)

def main() -> None:
    df = pd.read_csv(SRC_FILE, low_memory=False)

    # remove the row with “HTL Key” which is likely a header within the CSV
    df = df.rename(columns={"Unnamed: 0": "HTL_Key"})
    df = df[df["HTL_Key"].str.upper() != "HTL KEY"]

    # keep only the required columns
    df = df[["HTL_Key", "General Info"]].copy()

    # clean up columns
    df["HTL_Key"]      = df["HTL_Key"].apply(clean_htl_key)
    df["Google_Search"] = df["General Info"].apply(to_search_query)
    df = df.drop("General Info", axis=1) 
    
    df["Images"] = ""

    # save the result
    df.to_csv(DEST_FILE, index=False)
    print(f"Done! File saved at: {DEST_FILE.resolve()}")
    
main()

Done! File saved at: C:\Users\Dono\projects\python\escalera\hotels.cleaned.csv


Selenium scraper

In [3]:
def create_firefox_driver(headless:bool=False) -> webdriver.Firefox:
    """
    Creates and returns a configured Firefox driver
    """
    print("Launching Firefox browser with custom settings...") 

    options = FirefoxOptions()
    if headless:
        options.add_argument("--headless") 

    driver = webdriver.Firefox(options=options)
    print("Firefox browser launched.")
    return driver

def wait_for_element(driver: webdriver.Firefox, locator: tuple, timeout: int = 10):
    """
    Waiting for the element to become visible (up to timeout seconds)
    """
    print(f"Waiting for element {locator} (timeout={timeout}s)") 
    return WebDriverWait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )

In [ ]:
def get_image(driver: webdriver.Firefox, query: str, max_attempts: int = 3):
    links = []
    for attempt in range(1, max_attempts + 1):
        try:
            url = f"https://www.google.com/search?q=texas+{query}&tbm=isch"
            driver.get(url)
            
            # Wait for thumbnails to appear (class might change over time)
            wait_for_element(driver, (By.XPATH, "//h3[@class='ob5Hkd']/a"), timeout=4)
            thumbnails = driver.find_elements(By.XPATH, "//h3[@class='ob5Hkd']//img")

            for thumb in thumbnails[:5]:
                try:
                    # Click on the thumbnail to open a larger preview
                    driver.execute_script("arguments[0].click();", thumb)
                    time.sleep(2)
                    
                    # Output image in the preview has class 'n3VNCb'
                    preview = wait_for_element(driver, (By.XPATH, "//img[@class='sFlh5c FyHeAf iPVvYb']"), timeout=4)
                    src = preview.get_attribute('src')
                    if src and src.startswith('http'):
                        links.append(src)
                except:
                    continue
            
            # If we successfully collected image links, break the retry loop
            break
        
        except TimeoutException as te:
            print(f"[ attempt {attempt} ] Timeout: {te}, retrying...")
        except Exception as e:
            print(f"[ attempt {attempt} ] Unexpected error: {e}")
        finally:
            time.sleep(2)
    
    return links

In [ ]:
df = pd.read_csv(DEST_FILE, encoding='unicode_escape')
if 'Images' not in df.columns:
    df['Images'] = None
 
df['Images'] = df['Images'].astype(object)

In [6]:
driver = create_firefox_driver()

Launching Firefox browser with custom settings...
Firefox browser launched.


In [ ]:
# Iterate through empty images
for idx, row in df[df['Images'].isna()].iterrows():
    query = row['Google_Search']
    try:
        links = get_image(driver, query)
    except Exception as e:
        print(f"[{idx}] Greška pri skrejpanju '{query}': {e}")
        links = []
    
    # Save to DataFrame as a JSON string
    df.at[idx, 'Images'] = json.dumps(links, ensure_ascii=False)

    # SAVE the entire CSV immediately after each entry
    df.to_csv(DEST_FILE, index=False, encoding='utf-8')

# Close the browser
driver.quit()

NameError: name 'df' is not defined

Format the data for import into Snowflake

In [ ]:

df = pd.read_csv(DEST_FILE)
df = df.drop(columns=["Google_Search"])

def to_single_quote(val):
    # Convert the list to a string with single quotes
    try:
        lista = ast.literal_eval(val)
        return str(lista)  # This returns single quotes
    except Exception:
        return "[]"

df['Images'] = df['Images'].apply(to_single_quote)
 
df.to_csv(SNOWFLAKE_READY_FILE, index=False)